In [1]:
from google.colab import userdata
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
print("Key loaded:", PINECONE_API_KEY[:8] + "...")
print("Key length:", len(PINECONE_API_KEY))

Key loaded: pcsk_74e...
Key length: 75


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install -q pinecone sentence-transformers

In [4]:
import pandas as pd

train_df = pd.read_csv("/content/drive/MyDrive/contract-ai/train.csv")
val_df = pd.read_csv("/content/drive/MyDrive/contract-ai/val.csv")
test_df = pd.read_csv("/content/drive/MyDrive/contract-ai/test.csv")

all_clauses = pd.concat([train_df, val_df, test_df], ignore_index=True)
print(all_clauses.shape)

(13718, 4)


In [5]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(embedder.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

384


/tmp/ipykernel_13903/4255289260.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(embedder.get_sentence_embedding_dimension())


In [6]:
clause_texts = all_clauses["clause_text"].tolist()

embeddings = embedder.encode(
    clause_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(embeddings.shape)

Batches:   0%|          | 0/215 [00:00<?, ?it/s]

(13718, 384)


In [7]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "contract-clauses"

if index_name not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print("Index created")
else:
    print("Index already exists")

index = pc.Index(index_name)

Index created


In [8]:
print("key length:", len(PINECONE_API_KEY))

key length: 75


In [9]:
all_clauses = all_clauses.reset_index(drop=True)
all_clauses["clause_id"] = ["clause_" + str(i) for i in range(len(all_clauses))]

vectors_to_upload = []
for i, row in all_clauses.iterrows():
    vectors_to_upload.append({
        "id": row["clause_id"],
        "values": embeddings[i].tolist(),
        "metadata": {
            "contract": row["contract"],
            "clause_text": row["clause_text"][:1000],  # Pinecone metadata has size limits
            "label": row["label"],
        }
    })

print(len(vectors_to_upload), "vectors ready")
print(vectors_to_upload[0]["id"], vectors_to_upload[0]["metadata"]["label"])

13718 vectors ready
clause_0 Parties


In [10]:
batch_size = 100

for i in range(0, len(vectors_to_upload), batch_size):
    batch = vectors_to_upload[i:i + batch_size]
    index.upsert(vectors=batch)
    if i % 1000 == 0:
        print(f"Uploaded {i}/{len(vectors_to_upload)}")

print("Upload complete")

Uploaded 0/13718
Uploaded 1000/13718
Uploaded 2000/13718
Uploaded 3000/13718
Uploaded 4000/13718
Uploaded 5000/13718
Uploaded 6000/13718
Uploaded 7000/13718
Uploaded 8000/13718
Uploaded 9000/13718
Uploaded 10000/13718
Uploaded 11000/13718
Uploaded 12000/13718
Uploaded 13000/13718
Upload complete


In [11]:
stats = index.describe_index_stats()
print(stats)

DescribeIndexStatsResponse(dimension=384, total_vector_count=13718, metric='cosine', namespaces=1)
